# LEGO Rebrickable EDA + NLP (Colab skeleton)

## 1. Setup & imports

In [ ]:
!pip install pandas matplotlib seaborn plotly scikit-learn textblob nltk --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from textblob import TextBlob
import nltk
nltk.download('punkt')

## 2. Load data from GitHub

Adapt the `base_url` below if you upload files manually to Colab.

In [ ]:
base_url = "https://raw.githubusercontent.com/shannonlindsay/FabricCommunityContests/main/StarterFiles/"

files = {
    "colors": "colors.csv",
    "elements": "elements.csv",
    "inventories": "inventories.csv",
    "inventory_minifigs": "inventory_minifigs.csv",
    "inventory_sets": "inventory_sets.csv",
    "minifigs": "minifigs.csv",
    "part_categories": "part_categories.csv",
    "part_relationships": "part_relationships.csv",
    "parts": "parts.csv",
    "prices_data": "prices_data.csv",
    "sets": "sets.csv",
    "themes": "themes.csv"
}

dfs = {}
for name, fname in files.items():
    url = base_url + fname
    df = pd.read_csv(url)
    dfs[name] = df
    print(f"{name}: {df.shape}")

## 3. Data dictionary style overview

In [ ]:
for name, df in dfs.items():
    print(f"\n=== {name} ===")
    print(df.head())
    print(df.info())

## 4. Basic EDA on sets and themes

In [ ]:
sets = dfs["sets"]
themes = dfs["themes"]

# join themes onto sets (adjust column names if needed)
sets_themes = sets.merge(
    themes,
    left_on="theme_id",
    right_on="id",
    suffixes=("_set", "_theme")
)

# distribution of release year
plt.figure(figsize=(8, 4))
sns.histplot(sets["year"], bins=30, kde=False)
plt.title("Distribution of LEGO set release years")
plt.xlabel("Year")
plt.ylabel("Count")
plt.show()

# distribution of num_parts (trim 99th percentile to avoid crazy tails)
plt.figure(figsize=(8, 4))
sns.histplot(sets["num_parts"], bins=50, kde=False)
plt.xlim(0, sets["num_parts"].quantile(0.99))
plt.title("Distribution of set piece counts (trimmed)")
plt.xlabel("Pieces")
plt.ylabel("Count")
plt.show()

# average pieces per year
avg_pieces_year = (
    sets.groupby("year")["num_parts"]
    .mean()
    .reset_index()
    .rename(columns={"num_parts": "avg_num_parts"})
)

fig = px.line(
    avg_pieces_year,
    x="year",
    y="avg_num_parts",
    title="Average pieces per set over time"
)
fig.show()

## 5. Color analysis

In [ ]:
colors = dfs["colors"]
print("\nColors preview:")
print(colors.head())

# count how many distinct colors exist
print(f"Number of colors: {colors.shape[0]}")

## 6. Minifig analysis

In [ ]:
minifigs = dfs["minifigs"]
inv_minifigs = dfs["inventory_minifigs"]

print("\nMinifigs preview:")
print(minifigs.head())
print(inv_minifigs.head())

## 7. Lightweight NLP on text fields

In [ ]:
text_cols = {
    "sets": ["name"],
    "themes": ["name"],
    "parts": ["name"],
    "minifigs": ["name"]
}

for tbl, cols in text_cols.items():
    df = dfs[tbl].copy()
    for col in cols:
        df[col] = df[col].astype(str)

        # token count per name
        df["tok_count"] = df[col].apply(lambda x: len(nltk.word_tokenize(x)))
        print(f"\nNLP stats for {tbl}.{col}")
        print(df["tok_count"].describe())

        # simple keyword search: soccer/football terms
        keywords = ["football", "soccer", "stadium", "goal", "world cup"]
        mask = df[col].str.lower().str.contains("|".join(keywords))

        print(f"Found {mask.sum()} {tbl} rows mentioning soccer/football keywords")
        print(df.loc[mask, [col]].head(20))

## 8. Simple sentiment on set names (illustrative)

In [ ]:
sets["name"] = sets["name"].astype(str)
sets["sentiment"] = sets["name"].apply(lambda x: TextBlob(x).sentiment.polarity)

print("\nSample sentiment on set names:")
print(sets[["name", "sentiment"]].head())

plt.figure(figsize=(8, 4))
sns.histplot(sets["sentiment"], bins=30, kde=True)
plt.title("Sentiment distribution of set names")
plt.xlabel("Polarity")
plt.ylabel("Count")
plt.show()

## 9. Export summary tables

Export summary tables for later use (e.g., Power BI, Fabric).

In [ ]:
avg_pieces_year.to_csv("avg_pieces_year.csv", index=False)